In [50]:
import pandas as pd
import matplotlib.pyplot as plt
import pickle
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from xgboost import XGBRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score
from sklearn.model_selection import train_test_split

# Import the dataset

In [51]:
abc = pd.read_csv("Final_Log_dataset.csv")
df=abc.copy()

# Drop the unnecessary columns


In [52]:
df=df.drop(columns=["Unnamed: 0"])

# EDA Of Dataset

In [53]:
print(df.groupby("vehicle_type")["delivery_cost"].mean())
print(df.groupby("delivery_mode")["delivery_cost"].mean())
print(df.groupby("delivery_partner")[["delivery_cost","distance_km","package_weight_kg"]].mean())

vehicle_type
bike       11562.063568
ev bike     5917.383898
ev van     29336.732063
scooter     8811.797993
truck      71325.507567
van        46905.239927
Name: delivery_cost, dtype: float64
delivery_mode
express     45047.901533
standard    12731.043256
Name: delivery_cost, dtype: float64
                  delivery_cost  distance_km  package_weight_kg
delivery_partner                                               
amazon logistics   18947.863022   151.277167          24.926293
blue dart          37501.008940   149.738706          25.405332
delhivery          32316.722428   147.037904          25.467032
dhl                28940.906692   152.084404          24.891028
ecom express       49419.461180   151.066201          25.415422
ekart              10515.723040   151.566476          24.843609
fedex              21509.102972   148.778708          25.070298
shadowfax          18151.673154   151.037756          25.085355
xpressbees         42799.851297   150.974487          25.209827


# Categorise the data based on x & y .
 select the columns on there category

In [54]:
x = df.drop(columns=["delivery_cost"])
y = df["delivery_cost"]
categorical_features = ["delivery_partner","vehicle_type","delivery_mode"]
numeric_features = ["distance_km","package_weight_kg"]

# Encoode the variables
in the pipline 

In [56]:
preprocessor = ColumnTransformer([
        ("num",StandardScaler(),numeric_features),
        ("cat",OneHotEncoder(handle_unknown="ignore"),categorical_features),
    ])

# Initilize the Pipeline

In [57]:
regressor_xgboost = XGBRegressor(
    n_estimators=400,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42
)
regressor_rfor=RandomForestRegressor(n_estimators=100, random_state=42,max_depth=7)



model = Pipeline([
    ("preprocessing", preprocessor),
    ("regressor", regressor_rfor)
])


# Split the data for training & testing

In [58]:
x_train,x_test,y_train,y_test=train_test_split(x,y,test_size=0.2,random_state=42)

In [63]:
model.fit(x_train, y_train)
pred_val=model.predict(x_test)
print("the r2 score is ",r2_score(y_test,pred_val))
with open("delivery_price_model.pkl", "wb") as f:
    pickle.dump(model, f)

print("Model trained and saved")

the r2 score is  0.9157137162635332
Model trained and saved


In [ ]:
ohe = model.named_steps["preprocessing"].named_transformers_["cat"]
cat_features = ohe.get_feature_names_out(categorical_features)

all_features = numeric_features + list(cat_features)

importances = model.named_steps["regressor"].feature_importances_
feat_imp = pd.DataFrame({
    "feature": all_features,
    "importance": importances
}).sort_values(by="importance", ascending=False)

feat_imp.head(15).plot(
    kind="barh",
    x="feature",
    y="importance"
)

plt.gca().invert_yaxis()
plt.title("Feature Importance")
plt.show()

In [61]:
feat_imp["group"] = feat_imp["feature"].apply(
    lambda x: "distance" if "distance" in x else
              "weight" if "package_weight" in x else
              "vehicle" if "vehicle_type" in x else
              "mode" if "delivery_mode" in x else
              "partner"
)
print(feat_imp.groupby("group")["importance"].sum())

group
distance    0.164450
mode        0.283919
partner     0.161419
vehicle     0.366577
weight      0.023635
Name: importance, dtype: float64
